In [ ]:
# !pip install earthengine-api geemap geopandas

In [ ]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import json
from pathlib import Path

### *Reminder to swap to service account when integrating into app

In [ ]:
# only if first run or expired token
ee.Authenticate()

GEE_PROJECT = "nasa-eo-ccn"
ee.Initialize(project=GEE_PROJECT)

In [ ]:
# prototype hard coded sites
site_data = [
    {"site_id": "point_1", "name": "Site_A", "longitude": -91.04, "latitude": 29.47},
    {"site_id": "point_2", "name": "Site_B", "longitude": -90.23, "latitude": 29.62},
    {"site_id": "point_3", "name": "Site_C", "longitude": -89.83, "latitude": 29.49},
]

sites_df = pd.DataFrame(site_data)
sites_gdf = gpd.GeoDataFrame(
    sites_df,
    geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude),
    crs="EPSG:4326",
)

print(f"{len(sites_gdf)} sites loaded")
sites_gdf.head()

In [ ]:
# eventually load sites from CSV
# SITES_CSV = "path/to/ccn_sites.csv"
# sites_df = pd.read_csv(SITES_CSV)
# sites_gdf = gpd.GeoDataFrame(
#     sites_df,
#     geometry=gpd.points_from_xy(sites_df["longitude"], sites_df["latitude"]),
#     crs="EPSG:4326",
# )

In [ ]:
# convert gdf to ee.FeatureCollection
def gdf_to_ee_fc(gdf: gpd.GeoDataFrame) -> ee.FeatureCollection:
    """Convert a GeoDataFrame of points to an ee.FeatureCollection,
    preserving all attribute columns as properties."""
    features = []
    for _, row in gdf.iterrows():
        geom = ee.Geometry.Point([row.geometry.x, row.geometry.y])
        props = row.drop("geometry").to_dict()
        # convert numpy types
        props = {k: (v.item() if hasattr(v, "item") else v) for k, v in props.items()}
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)


points_fc = gdf_to_ee_fc(sites_gdf)
print("ee.FeatureCollection size:", points_fc.size().getInfo())

Querying EMIT:

In [ ]:
EMIT_COLLECTION = "NASA/EMIT/L2A/RFL"
DATE_START = "2023-01-01"
DATE_END = "2025-01-01"

emit_col = ee.ImageCollection(EMIT_COLLECTION).filterDate(DATE_START, DATE_END).filterBounds(points_fc)

n_images = emit_col.size().getInfo()
print(f"EMIT scenes matching filters: {n_images}")

In [ ]:
# metadata
def get_collection_info(collection: ee.ImageCollection) -> pd.DataFrame:
    """Pull date, scene ID, and cloud cover for every image in a collection."""
    info = collection.aggregate_array("system:index").getInfo()
    dates = collection.aggregate_array("system:time_start").getInfo()
    df = pd.DataFrame({"scene_id": info, "timestamp_ms": dates})
    df["date"] = pd.to_datetime(df["timestamp_ms"], unit="ms")
    return df.sort_values("date").reset_index(drop=True)


scenes_df = get_collection_info(emit_col)
print(f"Date range: {scenes_df['date'].min()} → {scenes_df['date'].max()}")
scenes_df.head(10)

Cloud masking:

In [ ]:
def mask_emit_clouds(img: ee.Image) -> ee.Image:
    """Mask cloudy pixels using the dilated_cloud_flag band.
    Flag == 0 indicates clear sky."""
    cloud_mask = img.select("dilated_cloud_flag").eq(0)
    return img.updateMask(cloud_mask)


# apply mask and build a mosaic composite
emit_masked = emit_col.map(mask_emit_clouds)
emit_mosaic = emit_masked.mosaic()

# verify band list
band_names = emit_mosaic.bandNames().getInfo()
print(f"Total bands in mosaic: {len(band_names)}")
print(f"First 10 bands: {band_names[:10]}")
print(f"Last 5 bands:  {band_names[-5:]}")

Server-side sampling:

In [ ]:
sampled_fc = emit_mosaic.sampleRegions(
    collection=points_fc,
    scale=60,
    geometries=True,
    tileScale=4,
)

print("Sampled features:", sampled_fc.size().getInfo())

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=sampled_fc,
    description="EMIT_Point_Extraction",
    folder="GEE_Exports",
    fileNamePrefix="emit_sampled_points",
    fileFormat="CSV",
)
task.start()
print(f"Export task submitted: {task.status()}")
print("Monitor at: https://code.earthengine.google.com/tasks")

Convert to df:

In [ ]:
sampled_df = geemap.ee_to_df(sampled_fc)
print(f"DataFrame shape: {sampled_df.shape}")
sampled_df.head()

In [ ]:
refl_cols = [c for c in sampled_df.columns if c.startswith("reflectance_")]
meta_cols = [c for c in sampled_df.columns if c not in refl_cols]

print(f"Reflectance bands extracted: {len(refl_cols)}")
print(f"Metadata columns: {meta_cols}")

In [ ]:
# wavelength lookup
first_img = ee.Image(emit_col.first())

try:
    wavelengths = first_img.get("wavelengths").getInfo()
    print(f"Retrieved {len(wavelengths)} center wavelengths from image metadata")
except Exception:
    import numpy as np

    wavelengths = np.linspace(381, 2493, len(refl_cols)).tolist()
    print(f"Using approximate wavelengths ({len(wavelengths)} bands)")

band_wavelength_map = {f"reflectance_{i}": wl for i, wl in enumerate(wavelengths)}

print(f"Band 0 → {wavelengths[0]:.1f} nm")
print(f"Band {len(wavelengths)-1} → {wavelengths[-1]:.1f} nm")

In [ ]:
spectra_long = sampled_df.melt(
    id_vars=meta_cols,
    value_vars=refl_cols,
    var_name="band",
    value_name="reflectance",
)
spectra_long["wavelength_nm"] = spectra_long["band"].map(band_wavelength_map)

print(f"Long-format shape: {spectra_long.shape}")
spectra_long.head()

In [ ]:
# eventually merge with CCN submission data
# CCN_FILE = "path/ccn_submission_data.csv"
# ccn_df = pd.read_csv(CCN_FILE)
#
# merged = spectra_long.merge(
#     ccn_df,
#     left_on="site_id",
#     right_on="site_id",
#     how="inner",
# )
# print(f"Merged records: {len(merged)}")
# merged.head()

In [ ]:
# save locally
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

sampled_df.to_csv(OUT_DIR / "emit_spectra_wide.csv", index=False)
spectra_long.to_csv(OUT_DIR / "emit_spectra_long.csv", index=False)

print(f"Saved to {OUT_DIR.resolve()}")

Example plot:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

for site_id, group in spectra_long.groupby("site_id"):
    group_sorted = group.sort_values("wavelength_nm")
    ax.plot(
        group_sorted["wavelength_nm"],
        group_sorted["reflectance"],
        label=site_id,
        linewidth=0.8,
    )

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Surface Reflectance")
ax.set_title("EMIT L2A Reflectance Spectra at CCN Sample Sites")
ax.legend()
ax.set_xlim(380, 2500)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

In [ ]:
Map = geemap.Map(center=[29.5, -90.3], zoom=8)

viz_params = {
    "bands": ["reflectance_51"],
    "min": 0,
    "max": 0.4,
}
Map.addLayer(emit_mosaic, viz_params, "EMIT Reflectance (band 51)")
Map.addLayer(points_fc, {"color": "red"}, "Sample Sites")

false_color = {
    "bands": ["reflectance_150", "reflectance_51", "reflectance_30"],
    "min": 0,
    "max": 0.35,
}
Map.addLayer(emit_mosaic, false_color, "EMIT False Color (SWIR/NIR/Red)", shown=False)

Map.add_layer_control()
Map

Optional extraction:

In [ ]:
def extract_single_scene(img: ee.Image) -> ee.FeatureCollection:
    """Sample a single EMIT image at all point locations,
    and attach the image date as a property."""
    date_str = img.date().format("YYYY-MM-dd")
    sampled = img.sampleRegions(
        collection=points_fc,
        scale=60,
        geometries=True,
        tileScale=4,
    )
    return sampled.map(lambda f: f.set("image_date", date_str))


# per_scene_fc = emit_masked.limit(5).map(extract_single_scene).flatten()
# per_scene_df = geemap.ee_to_df(per_scene_fc)
# print(f"Per-scene records: {per_scene_df.shape}")
# per_scene_df.head()